## function A*(Start, Goal):
1. Khởi tạo tập FRONTIER = {Start} với f(Start) = g(Start) + h(Start) = 0 + h(Start)

2. Khởi tạo tập REACHED = {}

3. TRONG KHI (FRONTIER không rỗng):

- a. Chọn trạng thái n từ FRONTIER có giá trị f(n) nhỏ nhất.

- b. NẾU n == Goal: TRẢ VỀ "Thành công" và truy xuất lại đường đi từ Start đến n.

- c. Loại bỏ n khỏi FRONTIER và thêm n vào REACHED.

- d. Với mỗi trạng thái m kề với n:

    - i. Tính toán chi phí thực tế mới:

        - g_new(m) = g(n) + cost(n, m)

    - ii. NẾU m đã nằm trong REACHED:

        - NẾU g_new(m) >= g(m) hiện tại: Bỏ qua trạng thái m (tệ hơn).

        - NGƯỢC LẠI: Xóa m khỏi REACHED và cập nhật lại g(m) = g_new(m).

    - iii. NẾU m đã nằm trong FRONTIER:

        - NẾU g_new(m) < g(m) hiện tại:

            - Cập nhật lại g(m) = g_new(m) và f(m) = g(m) + h(m).

            - Cập nhật lại đỉnh cha của m là n.

    - iv. NẾU m chưa có mặt trong FRONTIER và REACHED:

        - Gán g(m) = g_new(m). Tính f(m) = g(m) + h(m). Gán đỉnh cha của m là n. Thêm m vào FRONTIER.
4. TRẢ VỀ "Thất bại"

- Không tìm thấy đường đi tới đích.

In [1]:
import random

class Node:
    def __init__(self, room, parent, action, step, g, h):
        self.room = room
        self.parent = parent
        self.action = action
        self.step = step

        # g(n): số bước đã đi
        self.g = g

        # h(n): số ô bẩn còn lại
        self.h = h

        # f(n) = g(n) + h(n)
        self.cost = g + h


m = int(input("Nhập số hàng: "))
n = int(input("Nhập số cột: "))

room = []

print("Nhập ma trận:")

for i in range(m):
    row = list(map(int, input().split()))
    room.append(row)

start_x = random.randint(0, m - 1)
start_y = random.randint(0, n - 1)


def room_to_tuple(room):
    return tuple(tuple(row) for row in room)


def is_goal(room):
    for row in room:
        if 1 in row:
            return False

    return True


def print_room(room, x, y):
    for i in range(m):
        for j in range(n):
            if i == x and j == y:
                print("x", end=" ")
            else:
                print(room[i][j], end=" ")
        print()
    print()


def get_rules(x, y):
    moves = []

    if x > 0:
        moves.append("UP")

    if x < m - 1:
        moves.append("DOWN")

    if y > 0:
        moves.append("LEFT")

    if y < n - 1:
        moves.append("RIGHT")

    return moves


def move(room, x, y, action):
    new_room = [row[:] for row in room]
    new_x, new_y = x, y

    if action == "UP":
        new_x -= 1

    elif action == "DOWN":
        new_x += 1

    elif action == "LEFT":
        new_y -= 1

    elif action == "RIGHT":
        new_y += 1

    # hút bụi nếu ô mới có bụi
    if new_room[new_x][new_y] == 1:
        new_room[new_x][new_y] = 0

    return new_room, new_x, new_y


# h(n): số ô bẩn còn lại
def heuristic(room):
    cnt = 0
    for i in range(m):
        for j in range(n):
            if room[i][j] == 1:
                cnt += 1

    return cnt


def print_solution(node, start_x, start_y):
    path = []

    while node is not None:
        path.append(node)
        node = node.parent

    path.reverse()
    x, y = start_x, start_y
    print("Các bước làm sạch phòng:\n")

    for node in path:

        if node.action:
            _, x, y = move(node.parent.room, x, y, node.action)
            print(f"Bước {node.step}: {node.action} | g(n) = {node.g} | h(n) = {node.h} | f(n) = {node.cost}")
        else:
            print(f"Bước 0: Vị trí bắt đầu | g(n) = {node.g} | h(n) = {node.h} | f(n) = {node.cost}")

        print_room(node.room, x, y)

    print("Tổng số bước:", len(path) - 1)
    print("Cost cuối cùng:", path[-1].cost)


def A_star(start_room, start_x, start_y):

    start_room = [row[:] for row in start_room]

    # hút bụi ở vị trí bắt đầu
    if start_room[start_x][start_y] == 1:
        start_room[start_x][start_y] = 0

    frontier = []
    reached = {}

    start_g = 0
    start_h = heuristic(start_room)

    start_node = Node(start_room, None, None, 0, start_g, start_h)
    frontier.append((start_node, start_x, start_y))
    start_state = (room_to_tuple(start_room), start_x, start_y)
    reached[start_state] = start_node.cost

    while frontier:
        frontier.sort(key=lambda x: x[0].cost)

        current_node, x, y = frontier.pop(0)
        current_state = (room_to_tuple(current_node.room), x, y)
        if is_goal(current_node.room):
            return current_node

        # thêm vào reached
        reached[current_state] = current_node.cost

        # sinh trạng thái mới
        for act in get_rules(x, y):
            new_room, new_x, new_y = move(current_node.room, x, y, act)

            # g(n): số bước đã đi
            new_g = current_node.g + 1
            # h(n): số ô bẩn còn lại
            new_h = heuristic(new_room)
            # f(n)
            new_cost = new_g + new_h

            new_state = (room_to_tuple(new_room), new_x, new_y)
            skip = False

            # nếu state trong reached
            if new_state in reached and reached[new_state] <= new_cost:
                skip = True

            # nếu state trong frontier
            for node, fx, fy in frontier:
                state = (room_to_tuple(node.room), fx, fy)

                if state == new_state and node.cost <= new_cost:
                    skip = True
                    break

            if not skip:
                new_node = Node(new_room, current_node, act, current_node.step + 1, new_g, new_h)
                frontier.append((new_node, new_x, new_y))

    return None


print("Vị trí bắt đầu của máy hút bụi:", (start_x, start_y))
print()

result = A_star(room, start_x, start_y)

if result:
    print_solution(result, start_x, start_y)

else:
    print("Không tìm được lời giải")

Nhập ma trận:
Vị trí bắt đầu của máy hút bụi: (2, 0)

Các bước làm sạch phòng:

Bước 0: Vị trí bắt đầu | g(n) = 0 | h(n) = 4 | f(n) = 4
1 0 1 
0 1 0 
x 1 0 

Bước 1: RIGHT | g(n) = 1 | h(n) = 3 | f(n) = 4
1 0 1 
0 1 0 
0 x 0 

Bước 2: UP | g(n) = 2 | h(n) = 2 | f(n) = 4
1 0 1 
0 x 0 
0 0 0 

Bước 3: UP | g(n) = 3 | h(n) = 2 | f(n) = 5
1 x 1 
0 0 0 
0 0 0 

Bước 4: LEFT | g(n) = 4 | h(n) = 1 | f(n) = 5
x 0 1 
0 0 0 
0 0 0 

Bước 5: RIGHT | g(n) = 5 | h(n) = 1 | f(n) = 6
0 x 1 
0 0 0 
0 0 0 

Bước 6: RIGHT | g(n) = 6 | h(n) = 0 | f(n) = 6
0 0 x 
0 0 0 
0 0 0 

Tổng số bước: 6
Cost cuối cùng: 6
